# Welcome to your JupyterLab

This is your own private workspace. Nothing you do here affects anyone else — so feel free to experiment.

Work through this notebook once. It takes about five minutes.

## 1. How a notebook works

A notebook is a list of **cells**. Some cells hold text (like this one), others hold Python code.

To run a cell: click on it, then press **Shift+Enter**. The cell runs and the notebook moves on to the next one.

While a code cell is busy you will see `[*]` on its left. When it finishes, a number appears there instead, and anything the code printed shows up underneath.

Try it on the cell below.

In [ ]:
print("Hello! Everything is working.")

## 2. Saving your work

- JupyterLab **autosaves** your notebook every 30 seconds.
- Press **Ctrl+S** (Windows/Linux) or **Cmd+S** (Mac) to save right now.
- The **file browser** on the left shows your files. Double-click a file to open it.
- A **dot** on a tab (instead of an ×) means that file has changes that are not saved yet.

## 3. Your folders

| Folder | What it is for |
|---|---|
| `notebooks/` | Your own work. Make new notebooks here. |
| `submit/` | Your hand-in folder. Your teacher sees it instantly. |
| `shared/` | Handouts from your teacher. Read-only: copy a file into `notebooks/` to edit it. |

The next cell looks inside your home folder and lists what is there.

In [ ]:
from pathlib import Path

home = Path.home()
print(f"Your home folder is {home}\n")

notes = {
    "notebooks": "your own work",
    "submit": "hand-in folder — your teacher sees it",
    "shared": "handouts from your teacher (read-only)",
}
for item in sorted(home.iterdir()):
    if item.name.startswith("."):
        continue  # hidden settings files, not interesting
    kind = "folder" if item.is_dir() else "file"
    print(f"  {item.name:<16} {kind:<7} {notes.get(item.name, '')}")

## 4. Handing in work

When a piece of work is ready, save it (or copy it) into `submit/`. That is all — there is no "send" button. As soon as the file is in `submit/`, your teacher can see it.

Name files clearly so your teacher knows what they are and who they are from, for example `03-loops-priya.ipynb`.

To copy a file in the file browser: right-click it → **Copy**, open `submit/`, right-click → **Paste**.

The next cell hands in a tiny test file for you.

In [ ]:
import getpass
from datetime import datetime
from pathlib import Path

username = getpass.getuser()
submit = Path.home() / "submit"
submit.mkdir(exist_ok=True)

demo = submit / f"hello-from-{username}.txt"
stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
demo.write_text(f"Hello from {username}, written on {stamp}\n")

print(f"Created {demo.name} in your submit folder.")
print("Your teacher can now see this file in Submissions.")

## 5. Quick check (2 seconds)

This runs three libraries you will use a lot — **numpy**, **pandas** and **matplotlib** — and draws a tiny graph. If a picture appears under the cell, everything is working.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

x = np.linspace(0, 2 * np.pi, 200)
df = pd.DataFrame({"x": x, "sine": np.sin(x)})

fig, ax = plt.subplots(figsize=(5, 2.2))
ax.plot(df["x"], df["sine"])
ax.set_title("A sine wave, drawn with numpy, pandas and matplotlib")
ax.set_xlabel("x")
ax.set_ylabel("sin(x)")
plt.tight_layout()
plt.show()

print(f"numpy {np.__version__} | pandas {pd.__version__} | matplotlib {matplotlib.__version__}")

## 6. Full check (optional, ~30 s, loads PyTorch and TensorFlow)

Only run this if your teacher asks — it uses a lot of memory when everyone runs it at once.

It checks every library installed on this computer and prints `OK` or `FAIL` for each one.

In [ ]:
%run /srv/smoke_test.py

## 7. Radio (SDR) libraries

This computer also has two libraries for **software-defined radio** (SDR) — using a small USB receiver to pick up real radio signals:

- **pyrtlsdr** talks to cheap RTL-SDR USB dongles.
- **SoapySDR** is a general interface that works with many different radios (RTL-SDR included).

You do not need a radio to learn signal processing. The next cell makes its own signal, looks at its spectrum, and then checks whether a radio is plugged in.

*Using SoapySDR in a notebook of your own? Copy the `SOAPY_SDR_PLUGIN_PATH` line from the next cell to the top of it, so SoapySDR can find its radio drivers.*

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

# 1. Make a pretend radio signal: a 1 kHz tone plus some noise, 1 second at 48 000 samples/s.
fs = 48_000
t = np.arange(fs) / fs
rng = np.random.default_rng(0)
tone = np.sin(2 * np.pi * 1000 * t) + 0.3 * rng.standard_normal(t.size)

# 2. Look at its spectrum: the tall peak at 1 kHz is the tone, the flat floor is the noise.
freqs, power = signal.welch(tone, fs=fs, nperseg=2048)
fig, ax = plt.subplots(figsize=(5, 2.2))
ax.semilogy(freqs, power)
ax.set_xlim(0, 5000)
ax.set_title("Spectrum of a 1 kHz tone with noise")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("power")
plt.tight_layout()
plt.show()

# 3. Is a radio plugged in? (It is fine if not.)
# SoapySDR looks for its radio drivers only ONCE per kernel, so this line must run before the
# first search. Put it at the top of your own notebooks too, before the line that imports it.
os.environ.setdefault("SOAPY_SDR_PLUGIN_PATH", "/opt/conda/lib/SoapySDR/modules0.8")  # where the radio drivers live
try:
    import SoapySDR

    SoapySDR.setLogLevel(SoapySDR.SOAPY_SDR_FATAL)  # keep the driver's start-up chatter out of the way
    try:
        radios = SoapySDR.Device.enumerate()
    finally:
        SoapySDR.setLogLevel(SoapySDR.SOAPY_SDR_INFO)  # always put the normal messages back
    # A driver that loaded reports a version number; "" means SoapySDR never picked it up.
    drivers = [m for m in SoapySDR.listModules() if SoapySDR.getModuleVersion(m)]
    if radios:
        for radio in radios:
            print("SoapySDR found a radio:", radio)
    elif not drivers:
        print("SoapySDR: its radio drivers did not load, so it cannot see a dongle even if one is plugged in.")
        print("  Try the menu Kernel → Restart Kernel…, then run this cell again. Still no luck? Ask your teacher.")
    else:
        print("SoapySDR: No radio plugged in — that's fine for now.")
except Exception as problem:
    print("SoapySDR could not look for radios:", problem)

try:
    from rtlsdr import RtlSdr

    print("pyrtlsdr is ready. With an RTL-SDR dongle plugged in, RtlSdr() would open it.")
except Exception as problem:
    print("pyrtlsdr is not available:", problem)

## Tips

- **Stuck or frozen?** Use the menu **Kernel → Restart Kernel…** Your code stays; only the running Python is restarted.
- **Tab** completes names while you type. **Shift+Tab** with the cursor inside a function's brackets shows its help.
- **Run → Run All Cells** runs the whole notebook from the top.
- Stuck? Ask your teacher.

*Made with Classroom JupyterHub.*